# Eval100 test run: chart data as DataFrames

Loads the data behind every chart in `data/results/2026-11-18_claude_test_eval100_1/analysis/charts/`.
Each chart's DataFrame holds the same rows and columns that `Optimizer/charts.py` plots. Run from `notebooks/`.

In [ ]:
import json
from pathlib import Path
import pandas as pd

RUN = Path("../data/results/2026-11-18_claude_test_eval100_1")
ANALYSIS = RUN / "analysis"

meta = json.loads((RUN / "run_meta.json").read_text())
ARMS, LAST = list(meta["arms"]), int(meta["retries"])
FB = [a for a in ARMS if a != "control"]

t = {p.stem: pd.read_csv(p) for p in sorted(ANALYSIS.glob("*.csv"))}
{name: df.shape for name, df in t.items()}

## 01 Accuracy by attempt (`curves.csv`, Panacea line from `main.csv`)

In [ ]:
c, m = t["curves"], t["main"]
df_01 = c[c["scope"].isin(["all", "paired"])][["scope", "arm", "attempt", "n", "hidden_pass@1", "ci_low", "ci_high"]]
df_01_panacea = m[(m["system"] == "panacea") & (m["attempt"] == 0)][["system", "hidden_pass@1"]]
df_01

## 02 Gain vs control (`curves.csv`)

In [ ]:
df_02 = c[c["scope"].isin(["all", "paired"]) & (c["attempt"] > 0) & c["arm"].isin(FB)][
    ["scope", "arm", "attempt", "n", "gain_vs_control", "gain_ci_low", "gain_ci_high"]]
df_02

## 03 Rubric levels (`main.csv`)

In [ ]:
rows = [("zero_shot", 0)] + [(a, k) for a in ARMS for k in range(1, LAST + 1)]
if "panacea" in set(m["system"]):
    rows.append(("panacea", 0))
level_cols = [f"level_{lv}" for lv in range(5)] + ["level_none"]
df_03 = pd.concat([m[(m["system"] == s) & (m["attempt"] == k)] for s, k in rows])[["system", "attempt", "n"] + level_cols]
df_03

## 04–06 By group (`by_group.csv`, C + D tasks)

In [ ]:
bg = t["by_group"]
def by_group(grouping, scope="paired"):
    d = bg[(bg["grouping"] == grouping) & (bg["scope"] == scope) & bg["system"].isin(ARMS)]
    return d.rename(columns={"system": "arm"})[["group", "arm", "attempt", "n", "hidden_pass@1", "gain_vs_control"]]

df_04 = by_group("source_difficulty")
df_05 = by_group("attempt0_failure_type")
df_06 = by_group("test_format")
df_04

In [ ]:
df_05

In [ ]:
df_06

## 08 Repairs and breaks (`arm_outcomes.csv`)

In [ ]:
o = t["arm_outcomes"]
df_08 = o[["arm", "n_D", "n_C", "repair_rate", "false_fix_share", "break_rate", "regression_rate_per_retry"]]
df_08

## 09 Cost per fix (`arm_outcomes.csv`)

In [ ]:
df_09 = o[["arm", "cost_per_repair_usd", "tokens_per_repair", "extra_cost_per_extra_repair_usd"]]
df_09

## 10 Accuracy vs cost (`curves.csv` + `arm_outcomes.csv` + `main.csv`)

In [ ]:
allc = c[c["scope"] == "all"]
z = m[(m["system"] == "zero_shot") & (m["attempt"] == 0)].iloc[0]
pts = [("zero_shot", z["cost_usd"], allc[allc["attempt"] == 0]["hidden_pass@1"].iloc[0])]
oi = o.set_index("arm")
pts += [(a, oi.loc[a, "pipeline_cost_usd"],
         allc[(allc["arm"] == a) & (allc["attempt"] == LAST)]["hidden_pass@1"].iloc[0]) for a in ARMS]
pan = m[(m["system"] == "panacea") & (m["attempt"] == 0)]
if len(pan) and pan.iloc[0]["cost_usd"] > 0:
    pts.append(("panacea", pan.iloc[0]["cost_usd"], pan.iloc[0]["hidden_pass@1"]))
df_10 = pd.DataFrame(pts, columns=["system", "pipeline_cost_usd", "hidden_pass@1"])
df_10

## Other tables (no chart)
Every CSV is in `t`, e.g. `t["paired_tests"]`, `t["tasks"]`, `t["attempts"]`.

In [ ]:
t["paired_tests"]